In [ ]:
import os, math, json, gc
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

CLEAN_INPUT = 'data/clean/lichess_cleaned_minmoves6.csv'
OUTPUT_CSV = 'data/rewritten/lichess_rewritten.csv'
CHECKPOINT_DIR = 'rewritten_comments'
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

df = pd.read_csv(CLEAN_INPUT)
df.head()

# 2. Model loading

Phi-3-mini is loaded with 8-bit quantization to reduce memory usage.

In [ ]:
model_name = 'microsoft/Phi-3-mini-4k-instruct'
bnb_config = BitsAndBytesConfig(load_in_8bit=True, llm_int8_threshold=6.0, llm_int8_has_fp16_weight=False)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map='auto')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 3. Rewriting functions

Includes a concise prompt, batch inference, and incremental checkpoints.

In [ ]:
def build_prompt(text):
    return f"""
You rewrite chess comments into a short, objective description of the position.
Write 1–2 neutral sentences (max 40–50 words).
Use only the positional or tactical features explicitly mentioned: king safety, pawn structure, piece activity, key squares, initiative, or material balance (only if stated).
Do not invent details, moves, plans, evaluations, or emotions. Ignore subjective language.
If the comment gives no clear positional information, return "-".
Output only the final description.

Comment:
{text}
""".strip()

@torch.inference_mode()
def rewrite_batch(texts, batch_size=16, max_new_tokens=40, checkpoint_every=500, start_index=0, initial_outputs=None):
    if initial_outputs is None:
        initial_outputs = []
    outputs_all = initial_outputs.copy()
    batches = math.ceil(len(texts) / batch_size)
    last_ckpt = len(outputs_all)
    for i in tqdm(range(batches), desc='Rewriting'):
        batch = texts[i*batch_size:(i+1)*batch_size]
        prompts = [build_prompt(t) if isinstance(t, str) else '' for t in batch]
        inputs = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True).to(device)
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0, pad_token_id=tokenizer.eos_token_id)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        cleaned = []
        for full, prompt in zip(decoded, prompts):
            ans = full[len(prompt):].strip() if full.startswith(prompt) else full.strip()
            stop_markers = ['Comment', 'COMMENT', 'comment', 'Input comment', 'Description:']
            for mk in stop_markers:
                if mk in ans:
                    ans = ans.split(mk, 1)[0].strip()
                    break
            cleaned.append(ans)
        outputs_all.extend(cleaned)
        if checkpoint_every and len(outputs_all) - last_ckpt >= checkpoint_every:
            ckpt_path = os.path.join(CHECKPOINT_DIR, f'rewritten_0001-{len(outputs_all):05d}.json')
            with open(ckpt_path, 'w', encoding='utf-8') as f:
                json.dump(outputs_all, f, ensure_ascii=False, indent=2)
            last_ckpt = len(outputs_all)
        del inputs, outputs, decoded, cleaned, batch, prompts
        torch.cuda.empty_cache()
        gc.collect()
    return outputs_all

# 4. Run rewriting

Generate the `rewritten` column and save to CSV.

In [ ]:
import re

mask = df['annotation_expanded'].fillna('').apply(lambda x: bool(re.search(r'\d+\.', x)))
texts_to_rewrite = df.loc[mask, 'annotation_expanded'].fillna('').tolist()
rewritten_for_matching = rewrite_batch(texts_to_rewrite, batch_size=16, max_new_tokens=40, checkpoint_every=None, start_index=0, initial_outputs=[])
df['rewritten'] = df['annotation_expanded']
df.loc[mask, 'rewritten'] = rewritten_for_matching
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
print(f'Rewritings saved to: {OUTPUT_CSV}')

: 